In [1]:
import pandas as pd
import json
import ir_datasets
from src.data import DATA_DIR_PROCESSED, DATA_DIR_RAW
import os
from topic_gen.evaluate import QrelsEvaluator, CohenKappa, MeanAverageError, AreaUnderReceiver, ROSKendallTau, ROSRBO, binarize_qrels, load_runs_from_path
import ir_measures
from src.data import parse_file_names
from pathlib import Path

from topic_gen import logger
logger.setLevel("DEBUG")

In [2]:
# Load generated qrels from path
BASE_DIR = DATA_DIR_PROCESSED / "qrels"

predictions = []
names = []
metadata_records = []
for result in os.listdir(BASE_DIR):
    names.append(result)

    # metadata
    with open(os.path.join(BASE_DIR, result, "metadata.json")) as f:
        metadata = json.load(f)
    metadata_records.append(metadata)

    # predictions
    qrels = binarize_qrels(ir_measures.read_trec_qrels(
        os.path.join(BASE_DIR, result, "qrels.csv.gz")))
    predictions.append(qrels)

In [3]:
# Evaluate qrels
res = QrelsEvaluator.experiment(
    predictions=predictions,
    references=binarize_qrels(ir_datasets.load(
        "disks45/nocr/trec-robust-2004").qrels_iter()),
    measures=[CohenKappa(), MeanAverageError(), AreaUnderReceiver()],
    bootstrap=20,
    names=names)

[topic_gen] [INFO] (evaluate.py:249) Qrels in reference but not in predictions: 12 / 2939
[topic_gen] [INFO] (evaluate.py:249) Qrels in reference but not in predictions: 8 / 2943
[topic_gen] [INFO] (evaluate.py:249) Qrels in reference but not in predictions: 0 / 2951
[topic_gen] [INFO] (evaluate.py:249) Qrels in reference but not in predictions: 11 / 2940
[topic_gen] [INFO] (evaluate.py:249) Qrels in reference but not in predictions: 4 / 2947
[topic_gen] [INFO] (evaluate.py:249) Qrels in reference but not in predictions: 10 / 2941
[topic_gen] [INFO] (evaluate.py:249) Qrels in reference but not in predictions: 0 / 2951
[topic_gen] [INFO] (evaluate.py:249) Qrels in reference but not in predictions: 0 / 2951
[topic_gen] [INFO] (evaluate.py:249) Qrels in reference but not in predictions: 2 / 2949
[topic_gen] [INFO] (evaluate.py:249) Qrels in reference but not in predictions: 15 / 2936
[topic_gen] [INFO] (evaluate.py:249) Qrels in reference but not in predictions: 9 / 2942
[topic_gen] [INFO

In [4]:
# metadata table
metadata = pd.DataFrame(metadata_records)
metadata = metadata.join(pd.json_normalize(
    metadata["topics"]).add_prefix("topics_"))
metadata.drop(columns=["topics"], inplace=True)
metadata["topics_prompt"] = metadata["topics_prompt"].apply(
    lambda p: str(Path(p).stem) if pd.notnull(p) else "human")
metadata["prompt"] = metadata["prompt"].apply(lambda p: str(Path(p).stem))
metadata["model"] = metadata["model"].str.replace("-MT1000", "")
metadata["model"] = metadata["model"].str.replace("-MT100", "")

In [5]:
def format_score(row):
    return f"{row['value']:.2f} ± {row['ci']:.2f}"


table = pd.DataFrame(res)
table["score"] = table.apply(format_score, axis=1)
table = table.pivot(index="name", columns="measure",
                    values="score").reset_index()

In [6]:
table = table.merge(metadata, left_on="name", right_on="date")

## Prerequisite
### Can we reproduce the results from Thomas et al. with open models?
Yes! Open models outperform the results on GPT 4.1

In [7]:
table[(table["prompt"] == "-DNA-zero-shot")
      & (table["topics_prompt"] == "human")][["model", "prompt", "CohenKappa", "MeanAverageError", "AreaUnderReceiver"]]

,model,prompt,CohenKappa,MeanAverageError,AreaUnderReceiver
0,qwen3-30B-no-think,-DNA-zero-shot,0.75 ± 0.02,0.11 ± 0.01,0.89 ± 0.01
3,qwen3-14B-no-think,-DNA-zero-shot,0.72 ± 0.02,0.13 ± 0.01,0.85 ± 0.01
46,gpt-4.1,-DNA-zero-shot,0.64 ± 0.02,0.17 ± 0.01,0.81 ± 0.01
53,gpt-oss-120B,-DNA-zero-shot,0.70 ± 0.03,0.14 ± 0.01,0.84 ± 0.01
80,gpt-oss-120B,-DNA-zero-shot,0.69 ± 0.03,0.15 ± 0.01,0.84 ± 0.01


### Do LLM judges benefit from additional information in the topic?
Binary Setting:
- Für `GPT` scheint mehr Kontext schlechter zu sein. Für `Qwen` scheint mehr Kontext besser zu sein (mit ausnahmen...).
- Genereller ist der Effekt gering

In [ ]:
partial_annotation_prompts = [
    "-DNA-zero-shot", "-DNA-zero-shot-masked-title", "-DNA-zero-shot-masked-description", "-DNA-zero-shot-masked-narrative",
    "-DNA-zero-shot-masked-title-description", "-DNA-zero-shot-masked-title-narrative", "-DNA-zero-shot-masked-description-narrative"]

table["prompt"] = pd.Categorical(table["prompt"], partial_annotation_prompts)
table[table["topics_prompt"] == "human"].sort_values(by=["model", "prompt"])[
    ["model", "prompt", "CohenKappa", "MeanAverageError", "AreaUnderReceiver"]]

,model,prompt,CohenKappa,MeanAverageError,AreaUnderReceiver
46,gpt-4.1,-DNA-zero-shot,0.64 ± 0.02,0.17 ± 0.01,0.81 ± 0.01
53,gpt-oss-120B,-DNA-zero-shot,0.70 ± 0.03,0.14 ± 0.01,0.84 ± 0.01
80,gpt-oss-120B,-DNA-zero-shot,0.69 ± 0.03,0.15 ± 0.01,0.84 ± 0.01
86,gpt-oss-120B,-DNA-zero-shot-masked-title,0.69 ± 0.02,0.15 ± 0.01,0.84 ± 0.01
82,gpt-oss-120B,-DNA-zero-shot-masked-description,0.65 ± 0.03,0.17 ± 0.01,0.82 ± 0.01
83,gpt-oss-120B,-DNA-zero-shot-masked-narrative,0.64 ± 0.02,0.17 ± 0.01,0.82 ± 0.01
84,gpt-oss-120B,-DNA-zero-shot-masked-title-description,0.63 ± 0.02,0.18 ± 0.01,0.81 ± 0.01
85,gpt-oss-120B,-DNA-zero-shot-masked-title-narrative,0.69 ± 0.02,0.15 ± 0.01,0.83 ± 0.01
81,gpt-oss-120B,-DNA-zero-shot-masked-description-narrative,0.70 ± 0.02,0.14 ± 0.01,0.84 ± 0.01
3,qwen3-14B-no-think,-DNA-zero-shot,0.72 ± 0.02,0.13 ± 0.01,0.85 ± 0.01


## Alignment
RQ: How well aligne qrels based on generated topics with qrels based on the original qrels?

### Prompting Strategies
==Important: the columns ndocpos and ndocsneg state values even if they are ignored by the prompts.==

Prompts:
- Trec: query variants and relevant docs
- trec-query: only query
- trec-contrastive: query, relevant and irelevant documents


### Findings
- Judgments based on the original topics are allways better
- Some settings show a substantial drop in alignment, for example, for `qwen3-30B` on generated topics with the `trec` prompt with one query variant and one relevant document the Cohen's $\kappa$ agreement to human labels drops from the substantial agreement 0.75 to a moderate agreement of 0.52.
- More context allways helps
- Contrastive prompting is the best

In [21]:
prompt_sorter = ["human", "trec", "trec-query", "trec-contrastive"]
table["topics_prompt"] = pd.Categorical(
    table["topics_prompt"], prompt_sorter)

table[(table["prompt"] == "-DNA-zero-shot") & ((table["model"] == table["topics_model"]) | (table["topics_model"] == "trec assessors"))]\
    .sort_values(by=["model", "topics_prompt"], ascending=[True, True])[["topics_prompt", "model", "topics_nqueries", "topics_ndocspos", "topics_ndocsneg", "CohenKappa", "MeanAverageError", "AreaUnderReceiver"]]

,topics_prompt,model,topics_nqueries,topics_ndocspos,topics_ndocsneg,CohenKappa,MeanAverageError,AreaUnderReceiver
46,human,gpt-4.1,NaN,NaN,NaN,0.64 ± 0.02,0.17 ± 0.01,0.81 ± 0.01
53,human,gpt-oss-120B,NaN,NaN,NaN,0.70 ± 0.03,0.14 ± 0.01,0.84 ± 0.01
80,human,gpt-oss-120B,NaN,NaN,NaN,0.69 ± 0.03,0.15 ± 0.01,0.84 ± 0.01
72,trec,gpt-oss-120B,1.0,1.0,3.0,0.42 ± 0.02,0.32 ± 0.01,0.74 ± 0.01
73,trec,gpt-oss-120B,3.0,2.0,3.0,0.46 ± 0.02,0.29 ± 0.01,0.76 ± 0.01
74,trec,gpt-oss-120B,5.0,3.0,3.0,0.47 ± 0.03,0.28 ± 0.01,0.76 ± 0.01
75,trec-query,gpt-oss-120B,1.0,3.0,3.0,0.33 ± 0.02,0.38 ± 0.01,0.72 ± 0.01
76,trec-query,gpt-oss-120B,3.0,3.0,3.0,0.33 ± 0.01,0.38 ± 0.01,0.72 ± 0.01
77,trec-query,gpt-oss-120B,5.0,3.0,3.0,0.40 ± 0.02,0.33 ± 0.01,0.74 ± 0.01
78,trec-contrastive,gpt-oss-120B,1.0,1.0,1.0,0.51 ± 0.02,0.26 ± 0.01,0.77 ± 0.02


## Clarity
RQ: How well agree different annotators on the relevance lable using the generated toipics?
- Fleiss Kappa shows a moderate (>0.4) to just barely subtantial (>0.6) agreement across all three LLM annotators and topic prompts.


In [10]:
from typing import List
from statsmodels.stats import inter_rater as irr
import numpy as np
import krippendorff as kd


def agreement(dfs: List[pd.DataFrame]):
    df = dfs[0]
    rating_cols = ["relevance"]
    for idx, d in enumerate(dfs[1:]):
        df = df.merge(d, on=["query_id", "doc_id"],
                      suffixes=("", f"_r{idx+1}"))

        rating_cols.append(f"relevance_r{idx+1}")
    agg = irr.aggregate_raters(df[rating_cols])
    # fleiss kappa
    fleiss = irr.fleiss_kappa(agg[0], method='fleiss')

    arrT = np.array(df[rating_cols]).transpose()
    krippendorf = kd.alpha(arrT, level_of_measurement='nominal')

    return fleiss, krippendorf

In [ ]:
results = []
for i, df in table.groupby(by=["topics_model", "topics_prompt", "topics_nqueries", "topics_ndocsneg", "topics_ndocspos"]):
    row = df.iloc[0]
    name = row["topics_model"] + "_" + row["topics_prompt"] + \
        "-" + str(int(row["topics_nqueries"])) + \
        "-" + str(int(row["topics_ndocsneg"])) + \
        "-" + str(int(row["topics_ndocspos"]))
    qrels_dfs = []
    for run in df["name"]:
        qrel = pd.DataFrame(ir_measures.read_trec_qrels(
            os.path.join(BASE_DIR, run, "qrels.csv.gz")))
        qrels_dfs.append(qrel)

    fleiss, krippendorf = agreement(qrels_dfs)

    results.append({
        "name": name,
        "krippendorf": krippendorf,
        "fleiss": fleiss})

/tmp/ipykernel_2662550/380385313.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  for i, df in table.groupby(by=["topics_model", "topics_prompt", "topics_nqueries", "topics_ndocsneg", "topics_ndocspos"]):


In [20]:
pd.DataFrame(results)

,name,krippendorf,fleiss
0,gpt-oss-120B_trec-1-3-1,0.532897,0.532844
1,gpt-oss-120B_trec-3-3-2,0.570828,0.570779
2,gpt-oss-120B_trec-5-3-3,0.554660,0.554610
3,gpt-oss-120B_trec-query-1-3-3,0.567979,0.567906
4,gpt-oss-120B_trec-query-3-3-3,0.595185,0.595116
5,gpt-oss-120B_trec-query-5-3-3,0.565854,0.565780
6,gpt-oss-120B_trec-contrastive-1-1-1,0.596818,0.596749
7,gpt-oss-120B_trec-contrastive-5-3-3,0.637454,0.637391
8,qwen3-14B-no-think_trec-1-3-1,0.583660,0.583613
9,qwen3-14B-no-think_trec-3-3-2,0.552129,0.552079


### Cross Matrix

In [18]:
df = table[
    (table["prompt"] == "-DNA-zero-shot") &
    (table["topics_prompt"] == "trec-contrastive") &
    (table["topics_nqueries"] == 5) &
    (table["topics_ndocsneg"] == 3) &
    (table["topics_ndocspos"] == 3)
].copy()

df[["name", "CohenKappa", "model", "topics_model"]].pivot(
    index="model",
    columns="topics_model",
    values="CohenKappa")

topics_model,gpt-oss-120B,qwen3-14B-no-think,qwen3-30B-no-think
model,,,
gpt-oss-120B,0.51 ± 0.03,0.60 ± 0.02,0.52 ± 0.02
qwen3-14B-no-think,0.63 ± 0.02,0.70 ± 0.03,0.62 ± 0.02
qwen3-30B-no-think,NaN,0.74 ± 0.01,0.69 ± 0.03


## Distinguishability

RQ: How well can the generated topic help to distiguish between systems?


In [14]:
from ir_measures import nDCG, MAP, RBP

runs = load_runs_from_path(DATA_DIR_RAW / "trec-2004-runs")

In [ ]:
# Load generated qrels from path
BASE_DIR = DATA_DIR_PROCESSED / "qrels"

predictions = []
names = []
metadata_records = []
for result in os.listdir(BASE_DIR):

    # metadata
    with open(os.path.join(BASE_DIR, result, "metadata.json")) as f:
        metadata = json.load(f)

    # select only qrels
    if metadata["topics"]["prompt"] in {None, "../data/raw/prompts/trec-contrastive.yaml"}:
        metadata_records.append(metadata)
        # predictions
        qrels = ir_measures.read_trec_qrels(
            os.path.join(BASE_DIR, result, "qrels.csv.gz"))
        predictions.append(qrels)
        names.append(result)

In [17]:
reference = binarize_qrels(ir_datasets.load(
    "disks45/nocr/trec-robust-2004").qrels_iter())

In [ ]:
res = QrelsEvaluator.experiment(
    predictions=predictions,
    references=reference,
    measures=[ROSKendallTau(runs, measures=[nDCG@10, MAP@100]),
              ROSRBO(runs, measures=[nDCG@10, MAP@100], p=0.6, k=100)],
    # bootstrap=20,
    names=names
)

[topic_gen] [INFO] (evaluate.py:249) Qrels in reference but not in predictions: 11 / 2940
[topic_gen] [INFO] (evaluate.py:249) Qrels in reference but not in predictions: 0 / 2951
[topic_gen] [INFO] (evaluate.py:249) Qrels in reference but not in predictions: 5 / 2946
[topic_gen] [INFO] (evaluate.py:249) Qrels in reference but not in predictions: 0 / 2951
[topic_gen] [INFO] (evaluate.py:249) Qrels in reference but not in predictions: 4 / 2947
[topic_gen] [INFO] (evaluate.py:249) Qrels in reference but not in predictions: 0 / 2951
[topic_gen] [INFO] (evaluate.py:249) Qrels in reference but not in predictions: 16 / 2864
[topic_gen] [INFO] (evaluate.py:249) Qrels in reference but not in predictions: 8 / 2943
[topic_gen] [INFO] (evaluate.py:249) Qrels in reference but not in predictions: 2 / 2949
[topic_gen] [INFO] (evaluate.py:249) Qrels in reference but not in predictions: 0 / 2951
[topic_gen] [INFO] (evaluate.py:249) Qrels in reference but not in predictions: 0 / 2951
[topic_gen] [INFO] 

In [ ]:
table2 = pd.DataFrame(res)

In [ ]:
table2

,measure,value,name,missing
0,ROSKendallTau(nDCG@10),0.059883,2025-11-13_17:40:29,0
1,ROSKendallTau(AP@100),0.130609,2025-11-13_17:40:29,0
2,ROSRBO(nDCG@10),0.929365,2025-11-13_17:40:29,0
3,ROSRBO(AP@100),0.999447,2025-11-13_17:40:29,0
4,ROSKendallTau(nDCG@10),0.217348,2025-11-13_16:30:05,0
...,...,...,...,...
87,ROSRBO(AP@100),0.975622,2025-11-13_00:04:25,4
88,ROSKendallTau(nDCG@10),0.081234,2025-11-12_23:40:52,8
89,ROSKendallTau(AP@100),0.230359,2025-11-12_23:40:52,8
90,ROSRBO(nDCG@10),0.928015,2025-11-12_23:40:52,8


In [ ]:
table2.pivot(index="name", columns="measure", values="value").round(3)[
    ["ROSKendallTau(nDCG@10)", "ROSRBO(nDCG@10)", "ROSKendallTau(AP@100)", "ROSRBO(AP@100)"]]

measure,ROSKendallTau(nDCG@10),ROSRBO(nDCG@10),ROSKendallTau(AP@100),ROSRBO(AP@100)
name,,,,
2025-11-12_22:35:46,0.098,0.978,0.203,0.999
2025-11-12_22:49:21,0.194,0.924,0.157,0.994
2025-11-12_23:40:52,0.081,0.928,0.230,0.997
2025-11-12_23:58:35,0.250,0.923,0.146,0.994
2025-11-13_00:04:25,0.155,0.926,0.175,0.976
2025-11-13_00:10:30,0.263,0.928,0.281,0.994
2025-11-13_00:16:19,0.079,0.928,0.241,0.978
2025-11-13_00:22:18,0.136,0.974,0.148,0.998
2025-11-13_02:14:05,0.143,0.925,0.114,0.992
